# Station Stacking v10 - KAUS

Experimental notebook for `KAUS`.

This version keeps the v9 live-safe GFS/HRRR/NBM remaining-warmup data contract, trains only CatBoost with tuned Huber loss, disables ridge stacking, and writes artifacts to `data/calibration/station_stacking_v10`.


In [1]:
from pathlib import Path
import os
import sys
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", message="Skipping features without any observed values.*")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "calibration" / "station_stacking.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing src/calibration/station_stacking.py")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ["WEATHER_RESEARCH_INCLUDE_DIRECT_NBM"] = "1"

STATION_ID = "KAUS"
PROVIDERS = ("gfs", "hrrr", "nbm")
TREND_COLUMNS = [
    "observed_temp_change_last_1h_f",
    "observed_temp_change_last_3h_f",
    "observed_morning_warmup_rate_f_per_hour",
    "observed_high_so_far_change_since_9am_f",
]
TIMING_MODE = "same_day_11am_live_safe"
FAST_MODE = False
OPTUNA_TRIALS = 30
STACK_OPTUNA_TRIALS = 0
OPTUNA_STARTUP_TRIALS = 15
STACK_OPTUNA_STARTUP_TRIALS = 0
OPTUNA_METRIC = "mae_f"
OPTUNA_VERBOSE = True
MODEL_VERSION = "station_high_regressor_v10_catboost_huber"
PROJECT_ROOT


WindowsPath('D:/dev/weather-research')

In [2]:
import numpy as np
import pandas as pd

from src.export_station_stacking_v2_models import export_station_model_weights
from src.calibration.station_stacking import (
    StationStackingConfig,
    V10_DROPPED_FEATURE_COLUMNS,
    V10_FEATURE_COLUMNS,
    YEAR_SPLIT_EXPANDING_FOLDS,
    missing_model_dependencies,
    provider_availability,
    run_station_year_split_experiment,
)


## V10 Contract

`feature_version="v10"` keeps the v9 feature contract, uses `target_mode="remaining_warmup"`, and restricts the experiment to CatBoost with tuned Huber loss. Ridge stacking is disabled so the final exported model is the CatBoost base learner.


In [3]:
fold_spec = pd.DataFrame(
    [
        {
            "fold": fold.name,
            "train_start_year": fold.train_start_year,
            "train_end_year": fold.train_end_year,
            "validation_year": fold.validation_year,
        }
        for fold in YEAR_SPLIT_EXPANDING_FOLDS
    ]
)

fold_spec


,fold,train_start_year,train_end_year,validation_year
0,fold_2021_2023_to_2024,2021,2023,2024
1,fold_2021_2024_to_2025,2021,2024,2025


In [4]:
V10_FEATURE_COLUMNS, sorted(V10_DROPPED_FEATURE_COLUMNS)


(['v2_recent_heat_anomaly_f',
  'v2_recent_heat_momentum_f',
  'v2_morning_warmup_to_consensus_f',
  'v2_consensus_minus_7d_actual_f',
  'v2_spread_per_warmup_f',
  'v2_humidity_warmup_interaction',
  'v3_high_so_far_above_current_f',
  'v3_remaining_warmup_from_high_so_far_f',
  'v3_high_so_far_minus_lag_1d_f',
  'v3_high_so_far_minus_7d_actual_f',
  'v3_remaining_warmup_per_spread_f',
  'v3_humidity_remaining_warmup_interaction',
  'v4_forecast_precip_total_mean_mm',
  'v4_forecast_precip_total_max_mm',
  'v4_forecast_precip_total_spread_mm',
  'v4_forecast_precip_max_1h_mean_mm',
  'v4_forecast_precip_hours_mean',
  'v4_forecast_precip_intensity_mean',
  'v4_forecast_precip_intensity_max',
  'v4_any_forecast_precip',
  'v4_all_forecast_precip',
  'v4_observed_precip_any',
  'v4_observed_precip_recent_mm_est',
  'v4_forecast_total_minus_observed_recent_mm',
  'v4_forecast_observed_precip_match',
  'v4_forecast_wet_observed_dry',
  'v4_observed_wet_forecast_dry',
  'v4_precip_humidity

## Data Availability


In [5]:
availability = provider_availability(
    PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
)

availability.loc[availability["station_id"].eq(STATION_ID)]


,station_id,provider,row_count,first_contract_date,last_contract_date
3,KAUS,gfs,1983,2021-01-01,2026-06-10
4,KAUS,hrrr,1987,2021-01-01,2026-06-10
5,KAUS,nbm,1986,2021-01-01,2026-06-10


## Model Scores


In [6]:
missing_packages = missing_model_dependencies(("catboost",))
if missing_packages:
    raise ImportError(
        "Missing station-stacking ML packages: "
        + ", ".join(missing_packages)
        + ". Install them with: python -m pip install -r requirements.txt"
    )

config = StationStackingConfig(
    station_id=STATION_ID,
    project_root=PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
    fast_mode=FAST_MODE,
    optuna_trials=OPTUNA_TRIALS,
    stack_optuna_trials=STACK_OPTUNA_TRIALS,
    optuna_startup_trials=OPTUNA_STARTUP_TRIALS,
    stack_optuna_startup_trials=STACK_OPTUNA_STARTUP_TRIALS,
    optuna_metric=OPTUNA_METRIC,
    optuna_verbose=OPTUNA_VERBOSE,
    feature_version="v10",
    target_mode="remaining_warmup",
    base_model_methods=("catboost",),
    stack_enabled=False,
    hyperparameter_space="wide",
    year_split_folds=YEAR_SPLIT_EXPANDING_FOLDS,
    year_split_test_train_years=(2021, 2025),
    year_split_test_year=2026,
    output_dir=PROJECT_ROOT / "data" / "calibration" / "station_stacking_v10",
)

config.resolved_optuna_storage_path()


WindowsPath('D:/dev/weather-research/data/calibration/station_stacking_v10/KAUS_optuna.sqlite3')

In [7]:
result = run_station_year_split_experiment(config)
result.scoreboard


[I 2026-06-16 11:33:26,632] A new study created in RDB with name: KAUS_v10_remaining_warmup_base_catboost_mae_f
[I 2026-06-16 11:46:26,143] Trial 0 finished with value: 1.9253608367045798 and parameters: {'iterations': 2931, 'learning_rate': 0.001783588943232587, 'depth': 10, 'l2_leaf_reg': 0.9942958882300413, 'random_strength': 7.186216756161439, 'bagging_temperature': 12.184767612363034, 'border_count': 110, 'rsm': 0.6158971964095841, 'huber_delta': 1.5}. Best is trial 0 with value: 1.9253608367045798.
[I 2026-06-16 11:59:34,294] Trial 1 finished with value: 2.3116825406071566 and parameters: {'iterations': 801, 'learning_rate': 0.19767902101650964, 'depth': 12, 'l2_leaf_reg': 75.05056513807031, 'random_strength': 12.92821124806787, 'bagging_temperature': 4.276496242612171, 'border_count': 168, 'rsm': 0.4404450637901872, 'huber_delta': 1.0}. Best is trial 0 with value: 1.9253608367045798.
[I 2026-06-16 12:00:29,960] Trial 2 finished with value: 2.2554184018056613 and parameters: {'it

,period,method,count,mae_f,rmse_f
0,validation_2024_2025,catboost,729,1.681108,2.366744
1,validation_2024_2025,hrrr_raw,729,2.716582,3.534099
2,validation_2024_2025,gfs_raw,729,2.915723,3.874023
3,test_2026,catboost,137,1.709484,2.433309
4,test_2026,hrrr_raw,137,2.740739,3.273220
5,test_2026,gfs_raw,137,2.810295,3.512785


In [8]:
exported_weights = export_station_model_weights(
    project_root=PROJECT_ROOT,
    station_id=STATION_ID,
    artifact_dir=config.resolved_output_dir(),
    model_version=MODEL_VERSION,
    timing_mode=config.timing_mode,
    providers=tuple(config.providers),
    feature_version=config.effective_feature_version,
    optuna_metric=config.effective_optuna_metric,
    target_mode=config.effective_target_mode,
    base_model_methods=tuple(config.effective_base_model_methods),
    stack_enabled=config.stack_enabled,
    source_pipeline="notebooks/experiments/station_stacking_v10",
)

exported_weights.bundle_path, exported_weights.manifest_path


(WindowsPath('D:/dev/weather-research/data/calibration/station_stacking_v10/model_weights/KAUS_station_high_regressor_v10_catboost_huber.joblib'),
 WindowsPath('D:/dev/weather-research/data/calibration/station_stacking_v10/model_weights/KAUS_station_high_regressor_v10_catboost_huber.json'))

## V10 Feature Coverage


In [9]:
v10_feature_coverage = (
    result.features[V10_FEATURE_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)

v10_feature_coverage


,feature,coverage_pct
0,v2_spread_per_warmup_f,100.000000
1,v2_morning_warmup_to_consensus_f,100.000000
2,v3_remaining_warmup_from_high_so_far_f,100.000000
3,v3_high_so_far_above_current_f,100.000000
4,v2_humidity_warmup_interaction,100.000000
5,v4_observed_precip_recent_mm_est,100.000000
6,v4_forecast_wet_observed_dry,100.000000
7,v4_forecast_observed_precip_match,100.000000
8,v3_humidity_remaining_warmup_interaction,100.000000
9,v3_remaining_warmup_per_spread_f,100.000000


In [10]:
result.feature_columns.loc[result.feature_columns["feature"].isin(V10_FEATURE_COLUMNS)]


,feature,kind
20,observed_temp_change_last_1h_f,numeric
21,observed_temp_change_last_3h_f,numeric
22,observed_morning_warmup_rate_f_per_hour,numeric
23,observed_high_so_far_change_since_9am_f,numeric
163,v2_recent_heat_anomaly_f,numeric
164,v2_recent_heat_momentum_f,numeric
165,v2_morning_warmup_to_consensus_f,numeric
166,v2_consensus_minus_7d_actual_f,numeric
167,v2_spread_per_warmup_f,numeric
168,v2_humidity_warmup_interaction,numeric


## Dropped Feature Check


In [11]:
dropped_present = result.feature_columns.loc[
    result.feature_columns["feature"].isin(V10_DROPPED_FEATURE_COLUMNS)
]

dropped_present


,feature,kind


## Morning Trend Coverage


In [12]:
trend_coverage = (
    result.features[TREND_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)

trend_coverage


,feature,coverage_pct
0,observed_high_so_far_change_since_9am_f,100.000000
1,observed_temp_change_last_1h_f,99.949084
2,observed_temp_change_last_3h_f,99.949084
3,observed_morning_warmup_rate_f_per_hour,99.949084


## Rounded Within 1F Accuracy


In [13]:
preds = pd.concat(
    [
        result.validation_predictions.assign(period="validation_2024_2025"),
        result.test_predictions.assign(period="oof_2026"),
    ],
    ignore_index=True,
)

predicted_high = pd.to_numeric(preds["predicted_high_f"], errors="coerce")
preds["predicted_high_rounded_f"] = np.floor(predicted_high + 0.5)
preds["within_1f_after_round"] = (
    pd.to_numeric(preds["actual_high_f"], errors="coerce") - preds["predicted_high_rounded_f"]
).abs().le(1)

within_1f_accuracy_by_period = (
    preds
    .dropna(subset=["actual_high_f", "predicted_high_rounded_f"])
    .groupby(["period", "method"], as_index=False)
    .agg(
        count=("within_1f_after_round", "size"),
        within_1f_count=("within_1f_after_round", "sum"),
        within_1f_accuracy_pct=("within_1f_after_round", lambda x: x.mean() * 100),
    )
    .sort_values(["period", "within_1f_accuracy_pct"], ascending=[True, False])
)

within_1f_accuracy_by_period


,period,method,count,within_1f_count,within_1f_accuracy_pct
0,oof_2026,catboost,137,83,60.583942
3,oof_2026,nbm_raw,137,57,41.605839
1,oof_2026,gfs_raw,137,47,34.306569
2,oof_2026,hrrr_raw,137,39,28.467153
4,validation_2024_2025,catboost,729,443,60.768176
7,validation_2024_2025,nbm_raw,729,281,38.545953
6,validation_2024_2025,hrrr_raw,729,261,35.802469
5,validation_2024_2025,gfs_raw,729,229,31.412894


## Version Comparison


In [14]:
comparison_frames = []
for version, folder in [
    ("v1", PROJECT_ROOT / "data" / "calibration" / "station_stacking"),
    ("v2", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v2"),
    ("v3", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v3"),
    ("v4", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v4"),
    ("v5", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v5"),
    ("v6", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v6"),
    ("v7", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v7"),
    ("v8", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v8"),
    ("v9", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v9"),
    ("v10", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v10"),
]:
    path = folder / f"{STATION_ID}_year_split_scoreboard.csv"
    if path.exists():
        frame = pd.read_csv(path)
        frame["version"] = version
        comparison_frames.append(frame)

version_comparison = pd.concat(comparison_frames, ignore_index=True) if comparison_frames else pd.DataFrame()
if not version_comparison.empty:
    version_comparison = version_comparison.sort_values(["period", "mae_f", "version", "method"]).reset_index(drop=True)
version_comparison


,period,method,count,mae_f,rmse_f,version
0,test_2026,lightgbm,137,1.529158,2.006654,v8
1,test_2026,ridge_stack,137,1.557921,2.024451,v8
2,test_2026,ridge_stack,98,1.587308,2.183874,v7
3,test_2026,xgboost,137,1.594349,2.068159,v8
4,test_2026,xgboost,137,1.595938,2.087485,v9
...,...,...,...,...,...,...
100,validation_2024_2025,gfs_raw,660,4.616216,6.792032,v2
101,validation_2024_2025,gfs_raw,660,4.616216,6.792032,v3
102,validation_2024_2025,gfs_raw,660,4.616216,6.792032,v4
103,validation_2024_2025,gfs_raw,596,4.846561,7.070391,v1


## 2026 OOF Weather Brackets


In [15]:
result.bracket_metrics


,method,count,mae_f,rmse_f,bracket_accuracy_pct
0,catboost,137,1.709484,2.433309,39.416058
1,hrrr_raw,137,2.740739,3.273220,19.708029
2,gfs_raw,137,2.810295,3.512785,24.817518
